In [34]:
print("hello starting")

hello starting


In [35]:
import subprocess
from pathlib import Path

graphql_path = Path("AccountTransactions.graphql")
if not graphql_path.exists():
    raise FileNotFoundError(f"{graphql_path} not found")

graphql_registry = graphql_path.read_text()

#call ollama
import ollama

In [36]:
def analyse_graphql_query(user_query: str, graph_registry: str, model: str = "llama2"):
    prompt = f"""
You are an expert GraphQL API assistant. Your sole task is to generate complete GraphQL query requests and 
matching JSON response objects based on the provided GraphQL schema.

### GRAPHQL SCHEMA:
{graph_registry}

### CRITICAL RULES:
1. ONLY use fields, arguments, and enums explicitly defined in the schema above.
2. Every request must be a valid, syntactically correct GraphQL query string. The operation must semantically match the user query.
3. Every response must be a valid JSON object matching the exact structure and types requested in the GraphQL query.
4. Provide mock data values that are realistic for banking services (e.g., proper routing numbers, dates, masked account numbers, and currency strings).
5. Output your answer using exactly two markdown code blocks: the first labeled "```graphql" and the second labeled "```json". Do not add conversational filler.

### EXPECTED OUTPUT FORMAT:
### Request Query
```graphql
# GraphQL query goes here
```

### Response Object
```json
// Matching JSON response goes here
```

### USER INPUT:
{user_query}

"""
    #print(f"Prompt: {prompt}")
    return prompt


In [37]:
# Call the local Ollama instance
def call_ollama(prompt: str):
    response = ollama.chat(
        model='llama3.2',
        messages=[{
            'role': 'user',
            'content': prompt
        }]
    )

    # Print the response text
    #print(response['message']['content'])
    return response['message']['content']

In [38]:
user_query = "show me last 10 cash deposits in the branch"
prompt = analyse_graphql_query(user_query, graphql_registry)
response = call_ollama(prompt)
print(response)

```graphql
{
  getAccountBranchTransactions(
    accountId: "1234567890",
    transactionType: "CREDIT",
    transactionMode: "CASH"
    limit: 10
  )
}
```

```json
{
  "accountId": "1234567890",
  "transactionCount": 5,
  "transactions": [
    {
      "transactionId": "abc123def456",
      "date": "2023-03-20T14:30:00.000Z",
      "type": "CREDIT",
      "amount": 1000.00,
      "currency": "USD",
      "description": "Deposit from ATM",
      "status": "COMPLETED"
    },
    {
      "transactionId": "def789ghi012",
      "date": "2023-03-25T15:45:00.000Z",
      "type": "CREDIT",
      "amount": 500.00,
      "currency": "USD",
      "description": "Deposit from Mobile Banking",
      "status": "COMPLETED"
    },
    {
      "transactionId": "ghi345jklmno",
      "date": "2023-03-10T16:15:00.000Z",
      "type": "CREDIT",
      "amount": 2000.00,
      "currency": "USD",
      "description": "Deposit from Bank Branch",
      "status": "COMPLETED"
    },
    {
      "transactionId": 

In [43]:
user_query = "Forecast my spending for 8/31/2026"
prompt = analyse_graphql_query(user_query, graphql_registry)
response = call_ollama(prompt)
print(response)

```graphql
query {
  getForecastExpenses(
    accountId: "1234567890",
    targetDate: "2026-08-31"
  ) {
    accountId
    forecastDate
    projectedAmount
    confidenceScore
  }
}
```

```json
{
  "accountId": "1234567890",
  "forecastDate": "2026-08-31",
  "projectedAmount": 150.23,
  "confidenceScore": 0.85
}
```


In [41]:
user_query = "show me last 5 check deposits in the bank"
prompt = analyse_graphql_query(user_query, graphql_registry)
response = call_ollama(prompt)
print(response)

```graphql
query {
  getAccountBranchTransactions(
    accountId: "123456789",
    transactionMode: CHECK,
    limit: 5
  ) {
    transactions {
      transactionId
      date
      type
      amount
      currency
      description
      status
    }
  }
}
```

```json
{
  "transactions": [
    {"transactionId": "abc123", "date": "2023-02-20T14:30:00Z", "type": "DEBIT", "amount": 500.0, "currency": "USD", "description": "Rent payment", "status": "COMPLETED"},
    {"transactionId": "def456", "date": "2023-03-15T10:00:00Z", "type": "CREDIT", "amount": -1000.0, "currency": "USD", "description": "Salary deposit", "status": "PENDING"},
    {"transactionId": "ghi789", "date": "2023-04-01T16:30:00Z", "type": "DEBIT", "amount": 300.0, "currency": "USD", "description": "Groceries", "status": "COMPLETED"},
    {"transactionId": "jkl012", "date": "2023-05-20T12:45:00Z", "type": "CREDIT", "amount": -500.0, "currency": "USD", "description": "Utility bill payment", "status": "PENDING"},
    {"trans

In [42]:
user_query = "where is my money going?"
prompt = analyse_graphql_query(user_query, graphql_registry)
response = call_ollama(prompt)
print(response)

```graphql
{
  getAccountTransactions(
    accountId: "1234567890",
    transactionMode: TRANSACTION_MODE_CASH,
    limit: 10
  ) {
    transactions {
      amount
      date
      description
      status
      type
    }
    transactionCount
  }
}
```

```json
{
  "transactions": [
    {"amount": 100.45, "date": "2022-01-15", "description": "Groceries", "status": "COMPLETED", "type": "DEBIT"},
    {"amount": -50.20, "date": "2022-01-10", "description": "Rent payment", "status": "COMPLETED", "type": "CREDIT"}
  ],
  "transactionCount": 2
}
```
